# Notebook 1: Image Downsampling & Metadata Extraction

This notebook:
1. Reads TIFF images from a source folder
2. Downsizes them to a configurable max dimension (default 1536px)
3. Saves as compressed JPEG
4. Extracts TIFF/EXIF metadata into a CSV

**Note:** Downsampling to 50% was tested and found to have **no measurable impact** on model processing time or description quality. The primary bottleneck is model inference, not image size. Downsampled images are still useful for storage/transfer efficiency.

## Configuration
Set the paths and options below before running.

In [ ]:
import os

# ============================================================
# CONFIGURATION — Edit these paths before running
# ============================================================

# Input folder containing TIFF images
INPUT_DIR = r"path/to/your/input/images"  # e.g. r"C:\Downloads\Numbered Scans_1"

# Output folder for downsampled JPEGs and metadata CSV
OUTPUT_DIR = r"path/to/your/output/folder"  # e.g. r"C:\Downloads\Downsampled_Output"

# ============================================================
# RESIZE OPTIONS
# ============================================================

# Maximum pixel dimension (longest side). Choose one:
TARGET_SIZE = 1536   # Default — good balance of quality and file size
# TARGET_SIZE = 896  # Lighter images, faster transfer
# TARGET_SIZE = 2048 # Higher detail preservation

# JPEG quality settings
SAVE_FORMAT = "JPEG"
QUALITY = 90
OPTIMIZE = True

# Output CSV filename
METADATA_CSV = os.path.join(OUTPUT_DIR, "metadata_extracted.csv")

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Target size: {TARGET_SIZE}px max dimension")

## Imports

In [ ]:
import csv
from PIL import Image, TiffImagePlugin
from pathlib import Path

## Helper Functions

In [ ]:
def extract_metadata(img, filename, filepath):
    """Extract file-level, TIFF tag, and EXIF metadata from an open PIL image."""
    metadata = {"original_filename": filename}
    try:
        metadata["original_width"], metadata["original_height"] = img.size
        metadata["color_mode"] = img.mode
        metadata["format"] = img.format
        metadata["file_size_bytes"] = os.path.getsize(filepath)

        # TIFF TAG extraction
        if hasattr(img, "tag_v2"):
            for tag, value in img.tag_v2.items():
                key = f"TIFF_{TiffImagePlugin.TAGS.get(tag, tag)}"
                metadata[key] = str(value)

        # EXIF extraction
        exif = img.getexif()
        if exif:
            for tag, value in exif.items():
                tag_name = Image.ExifTags.TAGS.get(tag, tag)
                metadata[f"EXIF_{tag_name}"] = str(value)

    except Exception as e:
        metadata["metadata_error"] = str(e)

    return metadata


def process_image(filename):
    """Convert one TIFF to a downsampled JPEG and return its metadata."""
    try:
        in_path = os.path.join(INPUT_DIR, filename)
        base = os.path.splitext(filename)[0]
        out_path = os.path.join(OUTPUT_DIR, base + ".jpg")

        if os.path.exists(out_path):
            return None  # Already processed — skip

        with Image.open(in_path) as img:
            metadata = extract_metadata(img, filename, in_path)
            img = img.convert("RGB")
            img.thumbnail((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
            img.save(out_path, SAVE_FORMAT, quality=QUALITY, optimize=OPTIMIZE)

        return metadata

    except Exception as e:
        return {"original_filename": filename, "error": str(e)}

## Run Processing

In [ ]:
SUPPORTED_EXTENSIONS = ('.tif', '.tiff', '.jpg', '.jpeg', '.png', '.bmp')

files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
]

print(f"Found {len(files)} images. Beginning processing...\n")

all_rows = []

for idx, filename in enumerate(files, start=1):
    print(f"[{idx}/{len(files)}] Processing: {filename}")
    metadata = process_image(filename)
    if metadata is not None:
        all_rows.append(metadata)

print(f"\nProcessed {len(all_rows)} new images.")

## Save Metadata CSV

In [ ]:
if all_rows:
    fieldnames = sorted(set().union(*all_rows))
    with open(METADATA_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"Metadata saved to: {METADATA_CSV}")
else:
    print("No new images to process — all files already exist in output folder.")

print(f"\nDownsampled images saved to: {OUTPUT_DIR}")